In [1]:
import os
import cv2
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from tqdm import tqdm


In [2]:
# ---------------- CONFIG ----------------
ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"

OUT_ROOT = Path("dataset_pca3/images")

MODALITIES = ["reflec", "signal", "nearir", "range"]
SPLITS = ["train", "valid", "test"]
IMG_EXTS = [".png", ".jpg", ".jpeg"]

MAX_PIXELS = 200000   # sampling for PCA
np.random.seed(42)
# ----------------------------------------


In [3]:
def read_first_channel(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)

    if img.ndim == 3:
        img = img[:, :, 0]

    return img.astype(np.float32)


In [4]:
print("Collecting training pixels for PCA...")

train_dir = ROOT / MODALITIES[0] / "train"
image_files = [p for p in train_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

samples = []

for img_path in tqdm(image_files):
    stacked = []

    for mod in MODALITIES:
        mod_path = ROOT / mod / "train" / img_path.name
        img = read_first_channel(mod_path)
        stacked.append(img)

    stacked = np.stack(stacked, axis=-1)  # H x W x 4
    pixels = stacked.reshape(-1, 4)

    if len(pixels) > 1000:
        idx = np.random.choice(len(pixels), 1000, replace=False)
        pixels = pixels[idx]

    samples.append(pixels)

samples = np.concatenate(samples, axis=0)

if len(samples) > MAX_PIXELS:
    idx = np.random.choice(len(samples), MAX_PIXELS, replace=False)
    samples = samples[idx]

print("Total samples used for PCA:", samples.shape)


100%|██████████| 1367/1367 [01:05<00:00, 20.97it/s]

Total samples used for PCA: (200000, 4)


In [5]:
print("Fitting PCA...")

pca = PCA(n_components=3)
pca.fit(samples)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance captured:", pca.explained_variance_ratio_.sum())
def project_pca(img_stack, pca_model):
    H, W, C = img_stack.shape
    flat = img_stack.reshape(-1, C)
    projected = pca_model.transform(flat)
    projected = projected.reshape(H, W, 3)
    return projected

Fitting PCA...
Explained variance ratio: [0.6290127  0.28459573 0.07668317]
Total variance captured: 0.9902916


In [6]:
print("Creating PCA-3 fused dataset...")

for split in SPLITS:
    print(f"Processing split: {split}")

    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    ref_dir = ROOT / MODALITIES[0] / split
    image_files = [p for p in ref_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

    for img_path in tqdm(image_files):
        stacked = []

        for mod in MODALITIES:
            mod_path = ROOT / mod / split / img_path.name
            img = read_first_channel(mod_path)
            stacked.append(img)

        stacked = np.stack(stacked, axis=-1)  # H x W x 4

        projected = project_pca(stacked, pca)

        # Normalize each channel independently
        for c in range(3):
            channel = projected[:, :, c]
            channel -= channel.min()
            channel /= (channel.max() + 1e-6)
            projected[:, :, c] = channel

        projected = (projected * 255).astype(np.uint8)

        cv2.imwrite(str(out_dir / img_path.name), projected)

    print(f"Finished split: {split}")

print("PCA-3 dataset created successfully.")


Creating PCA-3 fused dataset...
Processing split: train


100%|██████████| 1367/1367 [00:40<00:00, 33.98it/s]


Finished split: train
Processing split: valid


100%|██████████| 390/390 [00:12<00:00, 30.88it/s]


Finished split: valid
Processing split: test


100%|██████████| 197/197 [00:06<00:00, 30.90it/s]

Finished split: test
PCA-3 dataset created successfully.


In [1]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="weighted_fusion.yaml",
    imgsz=1024,
    epochs=500,
    patience=80,        # early stopping
    batch=8,
    device=0,
    project="weighted_3pca",
    name="weighted_yolo11n_pca3",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025B330B2F90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [8]:
print(pca.components_)

[[    0.73732     0.60327     0.29304    0.080995]
 [    -0.4062    0.063066     0.90947   -0.062435]
 [   -0.52263      0.7945    -0.29491   -0.093054]]


In [1]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")
model.train(
    data="weighted_fusion.yaml",
    imgsz=1024,
    epochs=500,
    patience=40,
    batch=8,
    device=0,
    project="weighted_3pca_ms",
    name="weighted_yolo11n_pca3_ms",
    amp=False,
    augment=False,
    workers=0,
    scale=0.5,   # 👈 allow 50–150% scaling
)



Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=weighted_fusion.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021D92FC6C50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480